In [ ]:
import io
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

# Add src to Python path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from planetes.access import fetch_cutouts, find_frames
from planetes.cache import from_env

# Gaia DR3 2026 coordinates for Barnard's Star
ra_b, dec_b = 269.4461, 4.7703

# 1. Query frames (both QR2 and QR3)
df = find_frames(ra_b, dec_b, size_deg=0.04)

# 2. Select one frame per detector band (1 through 6)
selected_frames = df.groupby("detector", as_index=False).first().sort_values(by="detector")

# 3. Fetch cutouts
cache = from_env()
cutouts = fetch_cutouts(selected_frames, max_concurrent=6, cache=cache)

# 4. Render 6-panel figure
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(selected_frames.iterrows()):
    det = row["detector"]
    frame_id = row["frame_id"]
    key = f"cutouts/{frame_id}.fits"
    data_bytes = cutouts.get(key)
    ax = axes[i]

    if data_bytes:
        with fits.open(io.BytesIO(data_bytes)) as hdul:
            img_data = hdul[0].data if hdul[0].data is not None else hdul[1].data
            clean = img_data[np.isfinite(img_data)]
            vmin, vmax = np.percentile(clean, [5, 98]) if len(clean) > 0 else (0, 1)
            im = ax.imshow(img_data, origin="lower", cmap="magma", vmin=vmin, vmax=vmax)
            ax.set_title(f"Detector {det} ({row['release']})\n{frame_id}", fontsize=9)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Barnard's Star (2026 Epoch) — SPHEREx Cutouts", fontsize=14)
plt.tight_layout()
plt.show()

: 